# Day 5: Real Hardware: Transpilation, Noise, and Error Mitigation

**Clemson Quantum Club** · SC Quantathon v3 Bootcamp · [clemsonquantum.com](https://clemsonquantum.com)

In this notebook:
1. Setup: a snapshot of a real processor that runs offline
2. Meet the machine: qubits, connectivity, and calibration data
3. Transpilation: from your circuit to the chip's circuit
4. Where errors come from: decoherence, gate error, readout error
5. Simulating the machine, and comparing with a real run
6. Readout error mitigation
7. Zero-noise extrapolation
8. The same circuit on the live machine (optional)

> Cells marked **Your turn** have a few lines for you to fill in. Cells marked **Checkpoint** contain `assert` statements: if the cell runs without an error, the answer above it is correct. Solutions are in the companion solutions notebook.

## 1. Setup

Every earlier day ended with an optional cell that sent a circuit to a real IBM processor. Today the processor is the subject, so the notebook needs one that is always available. Qiskit ships **fake backends**: snapshots of real machines that carry the real calibration data, connectivity, and native gates, and that run a noisy simulation on your laptop. `FakeSherbrooke` is a snapshot of `ibm_sherbrooke`, a 127-qubit Eagle processor that was retired in July 2025; the snapshot is historical, the live machines today are Heron and Nighthawk processors, and everything the snapshot teaches carries over. Everything below runs against it, and section 8 shows how to point the same code at the live machine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp, hellinger_fidelity
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeManilaV2

np.set_printoptions(precision=3, suppress=True)

SHOTS = 1000
RUN_ON_HARDWARE = False     # section 8 only; False uses the recorded run

backend = FakeSherbrooke()          # a snapshot of ibm_sherbrooke, with its calibration data
rng = np.random.default_rng(7)
sampler = StatevectorSampler(seed=rng)

def next_seed():
    return int(rng.integers(2 ** 31 - 1))       # a fresh seed for every simulator run

def counts_of(qc, shots=SHOTS):
    return sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(name="scqv3")
    print("IBM account 'scqv3' loaded.")
except Exception as e:
    service = None
    print("No saved IBM account:", type(e).__name__, "(fine until section 8)")

## 2. Meet the machine

A superconducting processor is a chip of transmon qubits at about 15 mK, driven by microwave pulses. Three facts about it decide what a circuit can do: which qubits are physically connected (the **coupling map**), which gates the control electronics can play (the **native gates**), and how good each qubit and each gate is (the **calibration data**, remeasured every few hours). All three are attached to the backend object.

In [ ]:
target = backend.target
two_qubit_gate = [g for g in target.operation_names if g in ("ecr", "cz", "cx")][0]
edges = backend.coupling_map.get_edges()

print(f"{backend.name}: {backend.num_qubits} qubits")
print("native gates:      ", sorted(g for g in target.operation_names if g not in ("delay", "reset", "for_loop", "if_else", "switch_case")))
print("two-qubit gate:    ", two_qubit_gate)
print("coupled pairs:     ", len({tuple(sorted(e)) for e in edges}))

pairs = np.array(sorted({tuple(sorted(e)) for e in edges}))         # each coupled pair once, whichever direction the map lists
degree = np.bincount(pairs.ravel(), minlength=backend.num_qubits)
print("connections per qubit: min", degree.min(), " max", degree.max(), " mean", round(degree.mean(), 2))

Every qubit has a **T1**, the time for an excited $|1\rangle$ to decay to $|0\rangle$, a **T2**, the time for a superposition to lose its phase, and a **readout error**, the chance that a measurement reports the wrong bit. Every two-qubit gate has an error rate and a duration. The numbers below are the snapshot's calibration data.

In [ ]:
props = backend.properties()

print("qubit   T1 (us)   T2 (us)   readout error")
for q in range(6):
    print(f"{q:>4}   {props.t1(q) * 1e6:7.0f}   {props.t2(q) * 1e6:7.0f}   {props.readout_error(q):8.3%}")

t1_all = np.array([props.t1(q) for q in range(backend.num_qubits)]) * 1e6
t2_all = np.array([props.t2(q) for q in range(backend.num_qubits)]) * 1e6
ro_all = np.array([props.readout_error(q) for q in range(backend.num_qubits)])
gate_err = {pair: target[two_qubit_gate][pair].error for pair in target[two_qubit_gate]}
gate_dur = {pair: target[two_qubit_gate][pair].duration for pair in target[two_qubit_gate]}

print(f"\nmedian over {backend.num_qubits} qubits:  T1 {np.median(t1_all):.0f} us   T2 {np.median(t2_all):.0f} us   readout error {np.median(ro_all):.2%}")
print(f"{two_qubit_gate} gate: median error {np.median(list(gate_err.values())):.2%}, "
      f"worst {max(gate_err.values()):.1%}, duration {np.median(list(gate_dur.values())) * 1e9:.0f} ns")

plt.hist(t1_all, bins=25); plt.xlabel("T1 (us)"); plt.ylabel("qubits"); plt.title(f"T1 across {backend.name}"); plt.show()

# Calibration data is a set of measurements, not a specification. Physics says T2 <= 2 T1, and a readout error of
# 50% or a two-qubit error of 100% means "this qubit or pair is currently unusable"; a real chip has a few of each.
print(f"qubits whose measured T2 exceeds 2 T1: {int(np.sum(t2_all > 2 * t1_all))} of {backend.num_qubits}")
print(f"qubits with readout error >= 50%:     {int(np.sum(ro_all >= 0.5))}")
print(f"pairs with {two_qubit_gate} error >= 50%:          {sum(1 for e in gate_err.values() if e >= 0.5)}")

# Checkpoint: the typical qubit is healthy
assert np.median(t1_all) > 50 and np.median(t2_all) > 20 and np.median(ro_all) < 0.1

A single-qubit gate takes tens of nanoseconds and a two-qubit gate a few hundred, so a circuit of depth 20 runs in about ten microseconds, against coherence times of a few hundred microseconds. Decoherence is a small effect for short circuits and the dominant one for long ones; two-qubit gate error, at about a percent per gate, usually matters first.

### Your turn: the best pair

Find the coupled pair with the smallest two-qubit gate error and store it in `best_pair` as a tuple, together with its error in `best_error`. The dictionary `gate_err` maps pairs to errors.

In [ ]:
best_pair = None
best_error = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert best_pair is not None, "fill in the cell above"
assert best_pair in gate_err and np.isclose(best_error, min(gate_err.values()))
print(f"best pair {best_pair}: {two_qubit_gate} error {best_error:.3%}   (median {np.median(list(gate_err.values())):.3%})")

## 3. Transpilation

The circuits you write use $H$, CNOT, and whatever else is convenient, on qubits numbered from zero. The chip runs only its native gates, only between coupled pairs, on physical qubits it chooses. The **transpiler** bridges the gap with three jobs: choose a **layout** of your qubits onto physical qubits, **route** by inserting SWAPs wherever a two-qubit gate needs a pair that is not coupled, and **translate** every gate into native ones. Qiskit runs them as layout, routing, translation, and then an optimization pass. The result is an **ISA circuit** (instruction set architecture), the only kind a real backend accepts.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
bell_isa = pm.run(bell)

print("before:", dict(bell.count_ops()), " depth", bell.depth())
print("after: ", dict(bell_isa.count_ops()), " depth", bell_isa.depth())
print("physical qubits used:", sorted(bell_isa.layout.final_index_layout()))
bell_isa.draw("mpl", idle_wires=False)

$H$ became $R_z$ and $\sqrt X$ gates, CNOT became the native `ecr` plus single-qubit corrections, and the two logical qubits landed on two physical qubits that are coupled. The **optimization level** (0 to 3) controls how hard the transpiler works to shorten the result: cancelling adjacent gates, merging rotations, and searching for a layout that needs fewer SWAPs.

A few circuit tools help in reading what the transpiler produced. A `barrier` is a visual divider that also stops the optimizer from merging gates across it. `measure(q, c)` stores one qubit into one bit of a classical register you declared; `measure_all()` adds a register named `meas` and measures every qubit. `depth()` counts the layers of gates that cannot run at the same time, and `count_ops()` counts gates by name. The cell below isolates the translation step on a one-qubit circuit, $H, T, H$: no layout or routing is involved, only the rewriting into $R_z$ and $\sqrt X$.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.t(0)
qc.barrier()
qc.h(0)
qc.measure(0, 0)                    # one qubit into one declared classical bit
print("as written:  ", dict(qc.count_ops()), " depth", qc.depth())

pm_native = generate_preset_pass_manager(optimization_level=1, basis_gates=["rz", "sx", "x", "cx"])
native = pm_native.run(qc)
print("native gates:", dict(native.count_ops()), " depth", native.depth())

# Checkpoint: the same unitary, written in native gates only
no_meas = native.remove_final_measurements(inplace=False)
assert set(no_meas.count_ops()) <= {"rz", "sx", "x", "barrier"}
assert Operator(no_meas).equiv(Operator(qc.remove_final_measurements(inplace=False)))
native.draw("mpl")

In [ ]:
def ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for q in range(n - 1):
        qc.cx(q, q + 1)
    qc.measure_all()
    return qc

qc = ghz(5)
print("logical circuit:", dict(qc.count_ops()), " depth", qc.depth())
print("\nlevel   depth   two-qubit gates   total gates")
for level in range(4):
    isa = generate_preset_pass_manager(optimization_level=level, backend=backend, seed_transpiler=1).run(qc)
    ops = isa.count_ops()
    print(f"{level:>5}   {isa.depth():>5}   {ops.get(two_qubit_gate, 0):>15}   {sum(v for k, v in ops.items() if k not in ('measure', 'barrier')):>11}")

Level 1 produced the shortest circuit here and levels 2 and 3 a longer one: a higher level tries harder but is not guaranteed to be shorter, so the two-qubit gate count is the number to watch.

### Your turn: routing on a line

`FakeManilaV2` is a five-qubit machine whose qubits sit in a line: 0–1–2–3–4. The circuit below applies CNOTs from qubit 0 to every other qubit, which the line cannot do directly. Manila's native two-qubit gate is `cx`. Transpile it for Manila at optimization level 3, store the result in `star_isa`, and store the number of native two-qubit gates in `n_two_qubit`. It will be more than four: the extra ones are the SWAPs the router inserted, each of which costs three CNOTs.

In [ ]:
manila = FakeManilaV2()
print("Manila coupling map:", sorted(set(tuple(sorted(e)) for e in manila.coupling_map.get_edges())))

star = QuantumCircuit(5)
star.h(0)
for q in range(1, 5):
    star.cx(0, q)
star.measure_all()

star_isa = None
n_two_qubit = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert star_isa is not None and n_two_qubit is not None, "fill in the cell above"
assert set(star_isa.count_ops()) <= set(manila.target.operation_names) | {"barrier"}, "not in Manila's native gates"
assert n_two_qubit == star_isa.count_ops().get("cx", 0)
assert n_two_qubit > 4, "routing on a line must add two-qubit gates"
print(f"star circuit on a line: {n_two_qubit} native two-qubit gates for 4 logical CNOTs, depth {star_isa.depth()}")

## 4. Where errors come from

Three mechanisms account for nearly everything that separates a real run from the simulator. Each has a simple model, and Qiskit's `qiskit_aer.noise` module can attach any of them to a circuit, so each can be studied on its own before they are mixed.

### 4.1 Decoherence: T1 and T2

An excited qubit relaxes: after an idle time $t$, $|1\rangle$ has survived with probability $e^{-t/T_1}$. A superposition also dephases: the relative phase of $|+\rangle$ randomizes, so that after time $t$ a measurement in the $X$ basis returns $+$ with probability $\tfrac12(1 + e^{-t/T_2})$. The `thermal_relaxation_error` channel with the backend's own $T_1$ and $T_2$ for qubit 0 reproduces both curves.

In [ ]:
T1, T2 = props.t1(0), props.t2(0)
waits = np.linspace(0, 3 * T1, 13)

def idle_then_measure(prep_gates, wait, x_basis=False):
    qc = QuantumCircuit(1)
    for g in prep_gates:
        getattr(qc, g)(0)
    qc.append(thermal_relaxation_error(T1, T2, wait).to_instruction(), [0])
    if x_basis:
        qc.h(0)
    qc.measure_all()
    return AerSimulator().run(qc, shots=SHOTS, seed_simulator=next_seed()).result().get_counts()

p1_survive = [idle_then_measure(["x"], t).get("1", 0) / SHOTS for t in waits]

plt.plot(waits * 1e6, p1_survive, "o", label="simulated, 1000 shots")
plt.plot(waits * 1e6, np.exp(-waits / T1), "-", color="0.4", label=r"$e^{-t/T_1}$")
plt.xlabel("idle time (us)"); plt.ylabel("P(1) after preparing |1>"); plt.legend(frameon=False)
plt.title(f"T1 decay, T1 = {T1 * 1e6:.0f} us"); plt.show()

# Checkpoint: the survival probability follows the exponential
assert np.max(np.abs(np.array(p1_survive) - np.exp(-waits / T1))) < 0.06

### Your turn: the T2 curve

Prepare $|+\rangle$ (an `h` gate), idle for each time in `waits`, then measure in the $X$ basis. Store the fraction of $+$ outcomes (outcome `0` after the final $H$) in `p_plus`. It should follow $\tfrac12(1 + e^{-t/T_2})$, decaying from 1 toward $1/2$: a fully dephased qubit is a fair coin in every basis.

In [ ]:
p_plus = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint
assert p_plus is not None and len(p_plus) == len(waits), "fill in the cell above"
expected = 0.5 * (1 + np.exp(-waits / T2))
assert np.max(np.abs(np.array(p_plus) - expected)) < 0.06, "p_plus should follow (1 + exp(-t/T2))/2"
plt.plot(waits * 1e6, p_plus, "o", label="simulated"); plt.plot(waits * 1e6, expected, "-", color="0.4", label=r"$\frac{1}{2}(1 + e^{-t/T_2})$")
plt.xlabel("idle time (us)"); plt.ylabel("P(+) after preparing |+>"); plt.legend(frameon=False); plt.title(f"T2 dephasing, T2 = {T2 * 1e6:.0f} us"); plt.show()

### 4.2 Gate error

Each gate is a pulse that is slightly wrong. The simplest model is **depolarizing** noise: with probability $p$ the gate's output is replaced by the maximally mixed state, which is equally likely to read anything. Two CNOTs in a row cancel, so a chain of $2k$ CNOTs is the identity on paper. With gate error it is not: each gate leaves the state untouched with probability $1 - p$ and scrambles it otherwise, so after $k$ gates the chance that nothing has happened is $(1-p)^k$ and the survival of $|00\rangle$ is $\tfrac14 + \tfrac34(1-p)^k$, the $\tfrac14$ being the chance a scrambled state still reads `00`. This is a gate-repetition experiment. The standard hardware method, **randomized benchmarking**, does the same thing with sequences of random Clifford gates followed by an inverting gate, which averages coherent errors into exactly this depolarizing decay.

In [ ]:
p_gate = 0.01
depol = NoiseModel()
depol.add_all_qubit_quantum_error(depolarizing_error(p_gate, 2), ["cx"])
noisy_cx = AerSimulator(noise_model=depol)

ks = np.arange(0, 41, 4)
survival = []
for k in ks:
    qc = QuantumCircuit(2)
    for _ in range(k):
        qc.cx(0, 1)
    qc.measure_all()
    c = noisy_cx.run(qc, shots=SHOTS, seed_simulator=next_seed()).result().get_counts()
    survival.append(c.get("00", 0) / SHOTS)

plt.plot(ks, survival, "o", label=f"{p_gate:.0%} depolarizing error per CNOT")
plt.plot(ks, 0.25 + 0.75 * (1 - p_gate) ** ks, "-", color="0.4", label="1/4 + 3/4 (1 - p)^k")
plt.xlabel("number of CNOTs (an even number is the identity)"); plt.ylabel("P(00)"); plt.legend(frameon=False); plt.show()

# Checkpoint: the survival decays with the number of gates
assert survival[0] == 1.0 and survival[-1] < survival[0] - 0.15

### 4.3 Readout error

The last step, reading the qubit, has its own error: a $|0\rangle$ is sometimes reported as 1 and a $|1\rangle$ as 0, usually with different rates. The two rates make a $2\times2$ **confusion matrix**, whose columns are the prepared state and whose rows are the reported one. Readout error is the easiest to correct, because it acts after the circuit and the matrix can be measured directly (section 6).

In [ ]:
p01, p10 = 0.02, 0.05            # P(read 1 | prepared 0), P(read 0 | prepared 1)
readout = NoiseModel()
readout.add_all_qubit_readout_error(ReadoutError([[1 - p01, p01], [p10, 1 - p10]]))     # Aer's rows are the prepared state, the transpose of the confusion matrix M in section 6
noisy_read = AerSimulator(noise_model=readout)

for prep, label in [([], "|0>"), (["x"], "|1>")]:
    qc = QuantumCircuit(1)
    for g in prep:
        getattr(qc, g)(0)
    qc.measure_all()
    print(f"prepared {label}: {noisy_read.run(qc, shots=SHOTS, seed_simulator=next_seed()).result().get_counts()}")

## 5. Simulating the machine

`AerSimulator.from_backend` builds a noise model from the backend's calibration data, with thermal relaxation on every gate, depolarizing error at the measured rates, and each qubit's readout error, and runs the ISA circuit through it. It is the best offline prediction of what the machine will do. The comparison below is three-way: the ideal simulator, the noisy simulator, and Day 4's recorded run of the same Bell circuit, listed in the cell below with its processor and physical pair; that machine's two-qubit gate is `cz` rather than the snapshot's `ecr`. Expect the snapshot and the device to agree on the scale of the error, a percent or so of impossible outcomes, but not on the number: the snapshot is of an older machine, and the pair of qubits chosen on the live one was picked from that day's calibration table.

In [ ]:
noisy_backend = AerSimulator.from_backend(backend)

bell_ideal = counts_of(bell)
bell_noisy = noisy_backend.run(bell_isa, shots=SHOTS, seed_simulator=next_seed()).result().get_counts()
bell_real = {"00": 512, "11": 478, "01": 6, "10": 4}        # Day 4's run on ibm_kingston (Heron r2), physical qubits 55 and 59, 1000 shots

def impossible_fraction(counts):
    return (counts.get("01", 0) + counts.get("10", 0)) / sum(counts.values())

for name, c in [("ideal simulator", bell_ideal), ("noisy simulator", bell_noisy), ("real device", bell_real)]:
    print(f"{name:<16} {c}   impossible outcomes {impossible_fraction(c):.1%}   fidelity to ideal {hellinger_fidelity(bell_ideal, c):.3f}")

# Checkpoint: the noisy simulator and the device both show percent-scale impossible outcomes, and agree on that scale
assert 0.005 < impossible_fraction(bell_noisy) < 0.08, impossible_fraction(bell_noisy)
assert abs(impossible_fraction(bell_noisy) - impossible_fraction(bell_real)) < 0.06, (impossible_fraction(bell_noisy), impossible_fraction(bell_real))
plot_histogram([bell_ideal, bell_noisy, bell_real], legend=["ideal", "noisy simulator", "real device"], title="Bell state three ways")

## 6. Readout error mitigation

Readout error acts after the circuit, so it can be undone with linear algebra. Prepare each basis state in turn, measure it, and record the distribution of reported strings: that fills a matrix $M$ whose entry $M_{ij}$ is the probability of reading $i$ when $j$ was prepared. Any measured distribution $\vec p_{\rm meas}$ is then $M\vec p_{\rm true}$, and solving

$$\vec p_{\rm true} = M^{-1}\vec p_{\rm meas}$$

recovers the distribution the circuit would have produced with perfect readout. For two qubits $M$ is $4\times4$ and needs four calibration circuits. The corrected vector can have small negative entries from statistical noise; clipping and renormalizing is the usual fix.

### Your turn: the confusion matrix

Fill in `calibration_matrix(sim)`. For each of the four labels `"00"`, `"01"`, `"10"`, `"11"`, build a two-qubit circuit that prepares that state with `x` gates (remember qubit 0 is the rightmost character), transpile it with `pm_cal`, which pins it to the same physical qubits as the Bell circuit, run it on `sim`, and put the measured probabilities into column $j$ of a $4\times4$ array, rows ordered the same way.

In [ ]:
labels = ["00", "01", "10", "11"]
bell_physical = bell_isa.layout.final_index_layout()        # physical qubit for logical 0 and 1
pm_cal = generate_preset_pass_manager(optimization_level=0, backend=backend, initial_layout=bell_physical)
print("Bell circuit sits on physical qubits", bell_physical)

def calibration_matrix(sim, shots=SHOTS):
    M = np.zeros((4, 4))

    ### WRITE YOUR CODE BELOW HERE ###

    ### YOUR CODE FINISHES HERE ###
    return M

In [ ]:
# Checkpoint: columns are probability distributions, and the diagonal dominates
M = calibration_matrix(noisy_backend)
assert np.allclose(M.sum(axis=0), 1), "each column should sum to 1: fill in the cell above"
assert np.all(np.diag(M) > 0.8) and np.all(np.diag(M) < 1.0), np.diag(M)
print("confusion matrix M (rows: read, columns: prepared):\n", M)

In [ ]:
def mitigate(counts, M):
    p_meas = np.array([counts.get(k, 0) for k in labels], dtype=float)
    p_meas /= p_meas.sum()
    p_true = np.linalg.solve(M, p_meas)
    p_true = np.clip(p_true, 0, None)
    p_true /= p_true.sum()
    return dict(zip(labels, p_true))

assert np.allclose(M.sum(axis=0), 1), "complete calibration_matrix above first"
raw = {k: v / SHOTS for k, v in bell_noisy.items()}
fixed = mitigate(bell_noisy, M)
print("raw probabilities:      ", {k: round(raw.get(k, 0), 3) for k in labels})
print("mitigated probabilities:", {k: round(v, 3) for k, v in fixed.items()})
print(f"impossible outcomes: raw {impossible_fraction(bell_noisy):.1%} -> mitigated {fixed['01'] + fixed['10']:.1%}")

# Checkpoint: mitigation removes most of the readout contribution
assert fixed["01"] + fixed["10"] < impossible_fraction(bell_noisy)

What is left after mitigation is the part of the error that happened *inside* the circuit, mostly the two-qubit gate. Readout mitigation cannot touch that. The runtime primitives do this calibration for you: `EstimatorV2` with `resilience_level=1` applies a readout-error technique called TREX automatically.

## 7. Zero-noise extrapolation

Gate error cannot be calibrated away, but it can be **amplified** on purpose and then extrapolated out. Replace every CNOT by CNOT·CNOT·CNOT, which is the same gate on paper and three times the gate error in practice; do it again with five copies. An expectation value measured at noise scales $\lambda = 1, 3, 5$ traces a curve, and the value of that curve at $\lambda = 0$ is the estimate of the noiseless result. This is **zero-noise extrapolation** (ZNE). The example observable is $\langle ZZ\rangle$ of the Bell state, which is exactly 1 without noise.

In [ ]:
def folded_bell(n_copies):
    qc = QuantumCircuit(2)
    qc.h(0)
    for _ in range(n_copies):
        qc.cx(0, 1)
    qc.measure_all()
    return qc

def zz_from_counts(counts):
    total = sum(counts.values())
    return sum(v * (1 if k in ("00", "11") else -1) for k, v in counts.items()) / total

zne_noise = NoiseModel()
zne_noise.add_all_qubit_quantum_error(depolarizing_error(0.03, 2), ["cx"])     # a deliberately noisy CNOT
zne_sim = AerSimulator(noise_model=zne_noise)

noise_scales = np.array([1, 3, 5])
zz_values = []
for n_copies in noise_scales:
    c = zne_sim.run(folded_bell(n_copies), shots=20_000, seed_simulator=next_seed()).result().get_counts()
    zz_values.append(zz_from_counts(c))
print("noise scale:", noise_scales, "   <ZZ>:", np.round(zz_values, 4))

### Your turn: extrapolate to zero

Fit a straight line through the three points with `np.polyfit(noise_scales, zz_values, 1)` and evaluate it at $\lambda = 0$; store the result in `zz_linear`. Do the same with a quadratic and store it in `zz_quadratic`.

In [ ]:
zz_linear = None
zz_quadratic = None

### WRITE YOUR CODE BELOW HERE ###

### YOUR CODE FINISHES HERE ###

In [ ]:
# Checkpoint: both extrapolations beat the unmitigated value
assert zz_linear is not None and zz_quadratic is not None, "fill in the cell above"
assert abs(zz_linear - 1) < abs(zz_values[0] - 1), "the linear extrapolation should be closer to 1 than the raw value"
assert abs(zz_quadratic - 1) < abs(zz_values[0] - 1)
print(f"raw <ZZ> at scale 1: {zz_values[0]:.4f}   linear ZNE: {zz_linear:.4f}   quadratic ZNE: {zz_quadratic:.4f}   exact: 1")

lam = np.linspace(0, 5.5, 50)
plt.plot(noise_scales, zz_values, "o", ms=7, label="measured")
plt.plot(lam, np.polyval(np.polyfit(noise_scales, zz_values, 1), lam), "-", label="linear fit")
plt.plot(lam, np.polyval(np.polyfit(noise_scales, zz_values, 2), lam), "--", label="quadratic fit")
plt.axhline(1, color="0.4", lw=0.8); plt.xlabel("noise scale"); plt.ylabel("<ZZ>"); plt.legend(frameon=False); plt.show()

ZNE costs three runs instead of one and its answer is an extrapolation, not a measurement, so its error bars are larger. Its strength is that it needs no knowledge of the noise. On the runtime, `EstimatorV2` with `options.resilience.zne_mitigation = True` does the folding and fitting itself.

## 8. The same circuit on the live machine (optional)

Everything above used a snapshot. With an account and `RUN_ON_HARDWARE = True`, the cells below replace the snapshot with the least busy live processor and run the Bell circuit there, about 4 seconds of QPU time in all. One Sampler job carries five circuits: the Bell circuit and the four calibration circuits of section 6, transpiled onto the same physical pair, so that the confusion matrix is measured on the qubits that actually ran. A second job asks the runtime `EstimatorV2` for $\langle ZZ\rangle$ with its built-in readout mitigation and zero-noise extrapolation switched on, the two techniques of sections 6 and 7 done by the service. Three numbers to compare at the end: the exact $\langle ZZ\rangle = 1$, the snapshot's extrapolation of section 7, and the processor's.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler, EstimatorV2 as Estimator

job = job_e = None
if RUN_ON_HARDWARE and service is not None:
    live = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
    pm_live = generate_preset_pass_manager(optimization_level=3, backend=live)
    live_isa = pm_live.run(bell)
    live_phys = live_isa.layout.final_index_layout()
    print(f"{live.name}: native gates {dict(live_isa.count_ops())}, physical qubits {sorted(live_phys)}")

    # the four calibration circuits of section 6, pinned to the same physical pair
    pm_live_cal = generate_preset_pass_manager(optimization_level=0, backend=live, initial_layout=live_phys)
    cal_circuits = []
    for label in labels:
        qc = QuantumCircuit(2)
        for q, bit in enumerate(reversed(label)):
            if bit == "1":
                qc.x(q)
        qc.measure_all()
        cal_circuits.append(pm_live_cal.run(qc))
    job = Sampler(mode=live).run([live_isa] + cal_circuits, shots=SHOTS)
    print("Sampler job id:", job.job_id())

    # <ZZ> with the runtime's own mitigation: readout (TREX) and zero-noise extrapolation
    bell_isa_e = pm_live.run(bell.remove_final_measurements(inplace=False))
    zz_isa = SparsePauliOp("ZZ").apply_layout(bell_isa_e.layout)
    est = Estimator(mode=live)
    est.options.resilience_level = 1                      # readout mitigation (TREX)
    est.options.resilience.zne_mitigation = True          # zero-noise extrapolation
    job_e = est.run([(bell_isa_e, zz_isa)])
    print("Estimator job id:", job_e.job_id())
else:
    print("RUN_ON_HARDWARE is False or no account: section 5 already compared with a recorded device run.")

In [ ]:
if job is not None:
    try:
        results = job.result()
        live_counts = results[0].data.meas.get_counts()
        M_live = np.zeros((4, 4))
        for j in range(4):
            c = results[1 + j].data.meas.get_counts()
            for i, out in enumerate(labels):
                M_live[i, j] = c.get(out, 0) / SHOTS
        live_mitigated = mitigate(live_counts, M_live)
        zz_raw = sum(v * (1 if k in ("00", "11") else -1) for k, v in live_counts.items()) / SHOTS
        print("raw counts:      ", live_counts)
        print(f"impossible outcomes {impossible_fraction(live_counts):.1%}   fidelity to ideal {hellinger_fidelity(bell_ideal, live_counts):.3f}")
        print("confusion matrix diagonal on this pair:", np.round(np.diag(M_live), 3))
        print("readout mitigated:", {k: round(v, 3) for k, v in live_mitigated.items()})
        zz_mit = sum(v * (1 if k in ("00", "11") else -1) for k, v in live_mitigated.items())
        zz_runtime = float(job_e.result()[0].data.evs)
        print(f"\n<ZZ>: exact 1.0000   snapshot ZNE {zz_quadratic:.4f}   {live.name} raw {zz_raw:.4f}   readout mitigated {zz_mit:.4f}   runtime TREX + ZNE {zz_runtime:.4f}")
    except Exception as e:
        print("Job not finished or failed:", e)

## 9. Summary

- A backend carries a coupling map, a native gate set, and calibration data: T1, T2, readout error per qubit, and error and duration per two-qubit gate. Fake backends are offline snapshots with the same data.
- The transpiler lays out logical qubits onto physical ones, routes with SWAPs where the coupling map requires, and translates gates to native ones. A higher optimization level tries harder but is not guaranteed to be shorter; watch the two-qubit gate count. Only ISA circuits run on hardware.
- Decoherence: $|1\rangle$ survives as $e^{-t/T_1}$, a superposition keeps its phase as $e^{-t/T_2}$. Gate error compounds per gate. Readout error acts after the circuit.
- `AerSimulator.from_backend` predicts a real run from the calibration data; for the Bell state its impossible-outcome fraction lands within two percentage points of Day 4's device run (3.0% against 1.0%).
- Readout error mitigation inverts a measured confusion matrix. Zero-noise extrapolation amplifies gate noise by folding and extrapolates to zero. The runtime Estimator offers both as options.

## Further reading

- [Transpile](https://quantum.cloud.ibm.com/docs/en/guides/transpile) and [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques), IBM Quantum documentation
- [Building noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models) with Qiskit Aer
- [Fake backends](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider) in the runtime API reference
- P. Krantz et al., "A quantum engineer's guide to superconducting qubits," Applied Physics Reviews 6, 021318 (2019), for T1, T2, and gates in depth
- K. Temme, S. Bravyi, and J. M. Gambetta, "Error mitigation for short-depth quantum circuits," Phys. Rev. Lett. 119, 180509 (2017), the ZNE paper